ANALISI ESPLORATIVA

In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from datetime import datetime
import traceback
from scipy.signal import butter, sosfilt
import soundfile as sf
import pandas as pd
import os
import matplotlib.dates as mdates
import sys
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import noisereduce as nr
import shutil

c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

# --- CONFIGURAZIONE PERCORSI ---
INPUT_DIR = Path("AudioSamplesPreprocessed")   #Prima: Filtered
CSV_INPUT_PATH= Path("audio_samples_filtered_metadata.csv")
DATAFRAME_OUTPUT_PATH = Path("Dataframe")
NOISE_PATH = Path("noise.wav")

SAMPLE_RATE = 16000
FRAME_SIZE = 2048
HOP_LENGTH = 512
TARGET_DB = -1.0
target_amplitude = librosa.db_to_amplitude(TARGET_DB)
CUTOFF_FREQ = 100
NOISE_REDUCTION_PROPORTION = 1.0

def to_minutes(time_str):
    h, m = map(int, time_str.split(':'))
    return h * 60 + m

DATE_FORMAT = "%Y-%m-%dT%H:%M:%S%z"
TITLE_DATE_FORMAT = "%d-%m-%Y, %H:%M:%S"

# Creazione automatica delle cartelle
for d in [DATAFRAME_OUTPUT_PATH]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
if DATAFRAME_OUTPUT_PATH.exists():
    shutil.rmtree(DATAFRAME_OUTPUT_PATH)  # Rimuove la cartella e tutto il contenuto

DATAFRAME_OUTPUT_PATH.mkdir(parents=True) # La ricrea vuota

In [4]:
audio_df = pd.read_csv(CSV_INPUT_PATH)

audio_df["timestamp"] = pd.to_datetime(audio_df["timestamp"])

In [5]:
noise, _ = librosa.load(NOISE_PATH, sr=SAMPLE_RATE)

# 1. RIMOZIONE COMPONENTE DC 
noise = noise - np.mean(noise)

In [6]:
def get_octave_bands():
    """
    Genera i range (f_low, f_high) per bande d'ottava (fraction=1) 
    o terzi d'ottava (fraction=3).
    """
    # Frequenze centrali standard (base 10)
    # Per le ottave nel tuo range 0-8kHz
    f_min= 0
    f_max= 8000
    fraction=1
    f_center_base = [31.5, 63, 125, 250, 500, 1000, 2000, 4000, 8000]
    
    ranges = []
    # Fattore di larghezza banda: 2^(1/2N)
    factor = 2**(1 / (2 * fraction))
    
    for fc in f_center_base:
        flow = fc / factor
        fhigh = fc * factor
        # Filtriamo solo quelle che ricadono nel tuo spettro utile
        if fhigh > f_min and flow < f_max:
            ranges.append((max(flow, f_min), min(fhigh, f_max)))
            
    return ranges

In [7]:
def get_energy_in_single_frequency_range(S, freqs, frequency_range, aggregation_function="mean"):
    """
    Carica, processa e calcola l'energy media nelle bande Hz specificate.
    frequency_range: tuple di frequenze, es. (100, 500)
    """
    
    # ESTRAZIONE energy DAI RANGE DEFINITI
    f_min, f_max = frequency_range
    band_energy = 0
    
    # Trovo gli indici per questo specifico range
    idx = np.where((freqs >= f_min) & (freqs < f_max))[0]
    if len(idx) > 0:
        if aggregation_function=="mean":
            band_energy = np.mean(S[idx, :])
        elif aggregation_function=="median":
            band_energy = np.median(S[idx, :])
        else:
            band_energy = np.sum(S[idx, :])
    
    return band_energy

In [8]:
def q1(x): return x.quantile(0.25)
def q3(x): return x.quantile(0.75)

In [9]:
audio_df['timestamp'] = pd.to_datetime(audio_df['timestamp'])

audio_df['hour_decimal'] = (audio_df['timestamp'].dt.hour + 
                        audio_df['timestamp'].dt.minute / 60 + 
                        audio_df['timestamp'].dt.second / 3600)


audio_df['date'] = audio_df['timestamp'].dt.date
audio_df['day_name'] = audio_df['timestamp'].dt.day_name()
giorni_ita = {
    'Monday': 'Lunedì', 'Tuesday': 'Martedì', 'Wednesday': 'Mercoledì',
    'Thursday': 'Giovedì', 'Friday': 'Venerdì', 'Saturday': 'Sabato', 'Sunday': 'Domenica'
}
audio_df['day_it'] = audio_df['day_name'].map(giorni_ita)
ORDINE_GIORNI = ['Lunedì', 'Martedì', 'Mercoledì', 'Giovedì', 'Venerdì', 'Sabato', 'Domenica']
audio_df['day_it'] = pd.Categorical(audio_df['day_it'], categories=ORDINE_GIORNI, ordered=True)

audio_df['week_start'] = audio_df['timestamp'] - pd.to_timedelta(audio_df['timestamp'].dt.weekday, unit='D')
audio_df['week_end'] = audio_df['week_start'] + pd.to_timedelta(6, unit='D')
audio_df['periodo'] = (
    audio_df['week_start'].dt.strftime('%d %b') + 
    " - " + 
    audio_df['week_end'].dt.strftime('%d %b')
)

#audio_df['week_hour_continuous'] = (audio_df['timestamp'].dt.weekday * 24) + audio_df['hour_decimal']

# Ordina per tempo (fondamentale per il grafico)
#audio_df = audio_df.sort_values('hour_decimal')

In [10]:
# RANGE: 1400-3000, 4000-5000, 6500-8000 Hz
# RANGE (2): 1400-2200, 4000-6500
# RANGE (3): 1400-4500  Quello migliore
# RANGE (4): 1500-3000, 4000-5000, 5500-6500

In [11]:
# 1. Definiamo i passi richiesti
steps = [500, 1000, 2000]
min_hz = 0
max_hz = 8000

# 2. Creiamo una struttura per memorizzare i risultati dinamicamente
# Conterrà chiare del tipo: "0_500_norm", "500_1000_norm", ecc.
results = {}
total_files = len(audio_df)

# Inizializziamo le liste per ogni possibile range
octave_ranges = get_octave_bands()

# Aggiungiamo i range di ottava
for f_min, f_max in octave_ranges:
    
    # Creiamo un'etichetta leggibile arrotondando le frequenze
    # Esempio: "octave_707_1414"
    label = f"octave_{int(f_min)}_{int(f_max)}"
    
    # Inizializziamo le liste per i vari test che stai conducendo
    #results[f'energy_{label}_mean_with_normalized_audio'] = []
    #results[f'energy_{label}_median_with_normalized_audio'] = []
    #results[f'energy_{label}_sum_with_normalized_audio'] = []
    #results[f'energy_{label}_mean'] = []
    #results[f'energy_{label}_median'] = []
    results[f'energy_{label}'] = []

print("Ottave:", octave_ranges)

linear_ranges = []
for step in steps:
    for start in range(min_hz, max_hz, step):
        end = start + step
        if end <= max_hz:
            linear_ranges.append((start, end))
            #results[f'energy_{start}_{end}_mean_with_normalized_audio'] = []
            #results[f'energy_{start}_{end}_median_with_normalized_audio'] = []
            #results[f'energy_{start}_{end}_sum_with_normalized_audio'] = []
            #results[f'energy_{start}_{end}_mean'] = []
            #results[f'energy_{start}_{end}_median'] = []
            results[f'energy_{start}_{end}'] = []

print("Range lineari:", linear_ranges)

results['spectral_centroid'] = []

Ottave: [(22.273863607376246, 44.5477272147525), (44.54772721475249, 89.095454429505), (88.38834764831843, 176.7766952966369), (176.77669529663686, 353.5533905932738), (353.5533905932737, 707.1067811865476), (707.1067811865474, 1414.213562373095), (1414.2135623730949, 2828.42712474619), (2828.4271247461897, 5656.85424949238), (5656.8542494923795, 8000)]
Range lineari: [(0, 500), (500, 1000), (1000, 1500), (1500, 2000), (2000, 2500), (2500, 3000), (3000, 3500), (3500, 4000), (4000, 4500), (4500, 5000), (5000, 5500), (5500, 6000), (6000, 6500), (6500, 7000), (7000, 7500), (7500, 8000), (0, 1000), (1000, 2000), (2000, 3000), (3000, 4000), (4000, 5000), (5000, 6000), (6000, 7000), (7000, 8000), (0, 2000), (2000, 4000), (4000, 6000), (6000, 8000)]


In [12]:
# 3. Ciclo di processamento audio
for i, row in enumerate(audio_df.itertuples(), 1):
    filepath = INPUT_DIR / row.filename
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)

    #audio=preprocess(audio, offset_to_cut=0.5, normalize=False, noise_signal=noise)
    #normalized_audio=preprocess(audio, offset_to_cut=0.5, normalize=True)
    
    # SPETTROGRAMMA (Energia) 
    S = np.abs(librosa.stft(y=audio, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, window='hann'))**2
    #S_normalized_audio = np.abs(librosa.stft(y=normalized_audio, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, window='hann'))
    #S_dB = librosa.amplitude_to_db(S, ref=np.max)
    #S_normalized_audio_dB = librosa.amplitude_to_db(S_normalized_audio, ref=np.max)
    freqs = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=FRAME_SIZE)
    #S=S_dB
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=SAMPLE_RATE, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH)[0]
    mean_centroid = np.mean(spectral_centroid)
    results['spectral_centroid'].append(mean_centroid)
    
    # Header per il file corrente
    print(f"\n[{i}/{total_files}] File: {row.filename}")
    #print("-" * 50)
    
    for start, end in linear_ranges:
        # Nota: assumo che la tua funzione accetti una lista di tuple [(start, end)]
        current_range = (start, end)
        energy_median = get_energy_in_single_frequency_range(S, freqs, current_range, aggregation_function="median")
        results[f'energy_{start}_{end}'].append(energy_median)
    
    for f_min, f_max in octave_ranges:
        current_range = (f_min, f_max)
        label = f"octave_{int(f_min)}_{int(f_max)}"
        energy_median = get_energy_in_single_frequency_range(S, freqs, current_range, aggregation_function="median")
        results[f'energy_{label}'].append(energy_median)



[1/2554] File: audio_2026-02-23T09-26-59.wav

[2/2554] File: audio_2026-02-23T09-28-11.wav

[3/2554] File: audio_2026-02-23T10-37-03.wav

[4/2554] File: audio_2026-02-23T10-38-15.wav

[5/2554] File: audio_2026-02-23T10-39-26.wav

[6/2554] File: audio_2026-02-23T10-40-37.wav

[7/2554] File: audio_2026-02-23T10-53-07.wav

[8/2554] File: audio_2026-02-23T10-54-18.wav

[9/2554] File: audio_2026-02-23T10-56-41.wav

[10/2554] File: audio_2026-02-23T11-10-22.wav

[11/2554] File: audio_2026-02-23T11-11-34.wav

[12/2554] File: audio_2026-02-23T11-12-45.wav

[13/2554] File: audio_2026-02-23T11-13-56.wav

[14/2554] File: audio_2026-02-23T11-15-08.wav

[15/2554] File: audio_2026-02-23T11-40-57.wav

[16/2554] File: audio_2026-02-23T11-42-08.wav

[17/2554] File: audio_2026-02-23T11-43-20.wav

[18/2554] File: audio_2026-02-23T11-44-31.wav

[19/2554] File: audio_2026-02-23T11-45-43.wav

[20/2554] File: audio_2026-02-23T11-52-20.wav

[21/2554] File: audio_2026-02-23T11-58-59.wav

[22/2554] File: audio

In [13]:
campi_da_escludere = ['original_filename', 'sha256_hash']

base_df = audio_df.copy().drop(columns=campi_da_escludere, errors='ignore').reset_index(drop=True)

TIME_BIN = '5min'
PREFIX = "energy_"

# 1. Identifichiamo le chiavi del dizionario 'results'
target_keys = [k for k in results.keys() if k.startswith(PREFIX)]

# 2. Creiamo il DataFrame partendo dal filename
energy_df = base_df.copy().reset_index(drop=True)

energy_cols = []

# 3. Aggiungiamo le colonne corrispondenti
for key in target_keys:
    energy_df[key] = results[key]
    energy_cols.append(key)

In [14]:
# Salviamo il DataFrame nel dizionario con un nome descrittivo
df_filename="EnergiaMediana_PerRangeDiFrequenze.csv"
energy_df.to_csv(DATAFRAME_OUTPUT_PATH / df_filename, index=False)

In [15]:
ENERGY_BIN_AGGREGATION = 'median'

# Prepariamo il dataframe di base per il binning
# Usiamo floor('5min') sul timestamp per creare i bin temporali standard (00:00, 00:05, ecc.)
energy_df_per_binning = energy_df[['timestamp', 'day_it'] + energy_cols].copy()
energy_df_per_binning['time_bin'] = energy_df_per_binning['timestamp'].dt.floor(TIME_BIN).dt.time

energy_df_per_binning = energy_df_per_binning.sort_values(by=['day_it', 'time_bin'])

# 1. AGGREGAZIONE PER GIORNO DELLA SETTIMANA + BIN (es. Tutti i Lunedì alle 08:00)
# Raggruppiamo per il nome del giorno (italiano) e per l'orario del bin
energy_df_day_bin = energy_df_per_binning.groupby(['day_it', 'time_bin'])[energy_cols].agg([ENERGY_BIN_AGGREGATION,'std',q1,q3]).reset_index()

# Rinominiano le colonne in modo dinamico per gestire i nuovi livelli
new_cols = []
for col_name, stat in energy_df_day_bin.columns:
    if stat == '': # Per day_it e time_bin
        new_cols.append(col_name)
    elif stat == ENERGY_BIN_AGGREGATION:
        new_cols.append(col_name) # Mantiene il nome originale (es. energy_X)
    else:
        # Crea nomi tipo: energy_std_X, energy_q1_X, energy_q3_X
        # Assumendo che col_name inizi con 'energy_'
        suffix = col_name.split('_', 1)[1] if '_' in col_name else col_name
        new_cols.append(f"energy_{stat}_{suffix}")

energy_df_day_bin.columns = new_cols

day_filename = f"EnergiaMedianaPerBin5Minuti.csv"
energy_df_day_bin.to_csv(DATAFRAME_OUTPUT_PATH / day_filename, index=False)

In [16]:
# --- CALCOLO VARIAZIONE GIORNO/NOTTE ---

# Definiamo le fasce orarie (puoi cambiare questi orari come preferisci)
start_day = pd.to_datetime('06:00:00').time()
end_day = pd.to_datetime('22:00:00').time()

# Dividiamo tra giorno e notte
# Nota: la notte è "prima delle 06:00" O "dopo le 22:00"
is_day = (energy_df_day_bin['time_bin'] >= start_day) & (energy_df_day_bin['time_bin'] <= end_day)

# Calcoliamo la media per ogni banda (colonne energy) per Giorno e Notte
day_means = energy_df_day_bin[is_day].groupby('day_it')[energy_cols].mean()
night_means = energy_df_day_bin[~is_day].groupby('day_it')[energy_cols].mean()

# Calcolo variazione percentuale: ((Giorno - Notte) / Notte) * 100
variation_pct = ((day_means - night_means) / night_means) * 100
#variation_pct = day_means - night_means

# Reset index per avere 'day_it' come colonna e salvataggio
variation_pct = variation_pct.reset_index()

var_filename = f"VariazioneGiornoNotteDiEnergiaMediana.csv"
variation_pct.to_csv(DATAFRAME_OUTPUT_PATH / var_filename, index=False)


In [17]:
sc_df = base_df.copy().reset_index(drop=True)
sc_df["spectral_centroid"] = results["spectral_centroid"]

df_filename="SpectralCentroidMedio.csv"
sc_df.to_csv(DATAFRAME_OUTPUT_PATH / df_filename, index=False)

In [18]:
SPECTRAL_CENTROID_BIN_AGGREGATION = 'mean'

# Prepariamo il dataframe di base per il binning
# Usiamo floor('5min') sul timestamp per creare i bin temporali standard (00:00, 00:05, ecc.)
sc_df_per_binning = sc_df[['timestamp', 'day_it','spectral_centroid']].copy()
sc_df_per_binning['time_bin'] = sc_df_per_binning['timestamp'].dt.floor(TIME_BIN).dt.time

sc_df_per_binning = sc_df_per_binning.sort_values(by=['day_it', 'time_bin'])

# 1. AGGREGAZIONE PER GIORNO DELLA SETTIMANA + BIN (es. Tutti i Lunedì alle 08:00)
# Raggruppiamo per il nome del giorno (italiano) e per l'orario del bin
sc_df_day_bin = sc_df_per_binning.groupby(['day_it', 'time_bin'])[['spectral_centroid']].agg([SPECTRAL_CENTROID_BIN_AGGREGATION, 'std',q1,q3])

# 2. Rinomino le colonne MENTRE sono ancora un MultiIndex
sc_df_day_bin.columns = [
    "spectral_centroid_std" if s == 'std' else 
    "spectral_centroid_q1" if s == 'q1' else 
    "spectral_centroid_q3" if s == 'q3' else 
    "spectral_centroid" 
    for c, s in sc_df_day_bin.columns
]

# 3. Solo ORA faccio il reset_index
sc_df_day_bin = sc_df_day_bin.reset_index()

day_filename = f"SpectralCentroidMedioPerBin5Minuti.csv"
sc_df_day_bin.to_csv(DATAFRAME_OUTPUT_PATH / day_filename, index=False)

In [19]:
# --- CALCOLO VARIAZIONE GIORNO/NOTTE ---

# Definiamo le fasce orarie (puoi cambiare questi orari come preferisci)
start_day = pd.to_datetime('06:00:00').time()
end_day = pd.to_datetime('22:00:00').time()

# Dividiamo tra giorno e notte
# Nota: la notte è "prima delle 06:00" O "dopo le 22:00"
is_day = (sc_df_day_bin['time_bin'] >= start_day) & (sc_df_day_bin['time_bin'] <= end_day)

# Calcoliamo la media per ogni banda (colonne energy) per Giorno e Notte
day_means = sc_df_day_bin[is_day].groupby('day_it')["spectral_centroid"].mean()
night_means = sc_df_day_bin[~is_day].groupby('day_it')["spectral_centroid"].mean()

# Calcolo variazione percentuale: ((Giorno - Notte) / Notte) * 100
variation_pct = ((day_means - night_means) / night_means) * 100
#variation_pct = day_means - night_means

# Reset index per avere 'day_it' come colonna e salvataggio
variation_pct = variation_pct.reset_index()

var_filename = f"VariazioneGiornoNotteSpectralCentroid.csv"
variation_pct.to_csv(DATAFRAME_OUTPUT_PATH / var_filename, index=False)